# STGIPoser-syn vs Real IMU — TotalCapture_Real_60FPS

Runs the quarantined STGIPoser port (`external/stgiposer_totalcapture`, adapted `process_total_capture`) on one `TotalCapture_Real_60FPS/*.pkl` sequence and compares it against the **real** `acc`/`ori` from the same pkl.

**What the port does**
- Remaps pkl channels (GlobalPose order) to repo pelvis-first order with `TC_IMU_MASK = [5, 2, 3, 4, 0, 1]`.
- Drives FK from the AMASS pose npz (joint 37→23, 24 joints, `AMASS_ROT` root fix, `IG_JOINTS` zeroed).
- Synthesises vertex acceleration `vacc` with the wide-stencil `syn_acc` (PIP-style, zero-padded edges).
- **Bias-matches**: `imu_acc = acc_real + (vacc.mean − acc_real.mean)` — upstream's gravity/bias reconciliation. `imu_ori` stays real.
- Caveats: mean SMPL shape (no `betas`), `syn_acc` border frames are zero — metrics skip the first/last 4 frames.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import numpy as np
import torch
import roma
import matplotlib.pyplot as plt

from hmodel.bmodels.smpl import SMPL
from external.stgiposer_totalcapture import process_file

torch.set_printoptions(sci_mode=False)

# ── config ────────────────────────────────────────────────────────────
PKL_ROOT   = "D:/Projects/MemPoser/data/Total Capture/TotalCapture_Real_60FPS"
POSE_ROOT  = "D:/Projects/MemPoser/data/Total Capture/Pose"
MODEL_PATH = "D:/HModel/models"
SUBJECT    = "s1"
ACTION     = "walking1"        # pkl {SUBJECT}_{ACTION}.pkl ↔ Pose/{subject}/{action}_poses.npz
SAVE_FIGS  = False
FIGS_DIR   = "figs_stgiposer_vs_real"
SENSOR_SUBSET = None            # e.g. [0, 3] for Pelvis + Head only; None = all 6

SENSOR_NAMES = ["Pelvis", "L_LowLeg", "R_LowLeg", "Head", "L_LowArm", "R_LowArm"]
JOINT_MASK   = [0, 4, 5, 15, 18, 19]   # repo joints matching SENSOR_NAMES order
EDGE = 4                              # syn_acc border frames to exclude from metrics


In [ ]:
# ── run port on one sequence ───────────────────────────────────────
pkl_path = f"{PKL_ROOT}/{SUBJECT}_{ACTION}.pkl"
pose_path = f"{POSE_ROOT}/{SUBJECT}/{ACTION}_poses.npz"
gender = str(np.load(pose_path)["gender"])
print(f"seq: {SUBJECT}_{ACTION}, gender={gender}")

smpl = SMPL(path=MODEL_PATH, gender=gender)
out = process_file(pkl_path, pose_path, smpl)
for k, v in out.items():
    print(f"{k:15s} {tuple(v.shape)} {v.dtype}")

W = out["imu_acc"].shape[0]
print(f"frames: {W}")
print("bias shift per sensor (vacc.mean \u2212 acc_real.mean), m/s\u00b2:")
for si, sname in enumerate(SENSOR_NAMES):
    d = out["acc_bias_shift"][si]
    print(f"  {sname:9s} [{d[0]:+7.3f}, {d[1]:+7.3f}, {d[2]:+7.3f}]")

# Orientation traces in a common bone frame (rotvec, rad)
grot_6 = out["grot"][:, JOINT_MASK]          # FK joint rotations, repo order
ori_real = out["imu_ori"]                    # real orientations, repo order
ori_syn = roma.rotmat_to_rotvec(grot_6.reshape(-1, 3, 3)).reshape(W, 6, 3)
ori_rr = roma.rotmat_to_rotvec(ori_real.reshape(-1, 3, 3)).reshape(W, 6, 3)


In [ ]:
# ── ACC metrics: vacc vs matched real, vacc vs raw real (edges excluded) ──
sl = slice(EDGE, W - EDGE)
vacc = out["vacc"][sl]
acc_matched = out["imu_acc"][sl]
acc_raw = out["imu_acc_raw"][sl]

print(f"{'target':22s} {'sensor':9s} | {'RMSE-x':>8s} {'RMSE-y':>8s} {'RMSE-z':>8s} | {'bias-x':>8s} {'bias-y':>8s} {'bias-z':>8s}  m/s\u00b2")
print("-" * 100)
for tname, t in (("vacc-vs-matched", acc_matched), ("vacc-vs-raw", acc_raw)):
    for si, sname in enumerate(SENSOR_NAMES):
        err = vacc[:, si] - t[:, si]
        rmse, bias = err.pow(2).mean(dim=0).sqrt(), err.mean(dim=0)
        print(f"{tname:22s} {sname:9s} | {rmse[0]:8.4f} {rmse[1]:8.4f} {rmse[2]:8.4f} | "
              f"{bias[0]:+8.4f} {bias[1]:+8.4f} {bias[2]:+8.4f}")
    print("-" * 100)

# ── ORI metrics: FK grot vs real ori ──────────────────────────────────
R = grot_6.transpose(-1, -2) @ ori_real
tr = R.diagonal(dim1=-2, dim2=-1).sum(-1).clamp(-1, 3)
geod = torch.rad2deg(torch.acos(((tr - 1) / 2).clamp(-1, 1)))   # (W, 6) deg
print(f"{'sensor':9s} | {'geod-mean\u00b0':>10s} {'geod-med\u00b0':>10s} | {'RMSE-x':>8s} {'RMSE-y':>8s} {'RMSE-z':>8s}  rad")
print("-" * 80)
for si, sname in enumerate(SENSOR_NAMES):
    err = ori_syn[:, si] - ori_rr[:, si]
    rmse = err.pow(2).mean(dim=0).sqrt()
    print(f"{sname:9s} | {geod[:, si].mean():10.2f} {geod[:, si].median():10.2f} | "
          f"{rmse[0]:8.4f} {rmse[1]:8.4f} {rmse[2]:8.4f}")


In [ ]:
# ── ACC plots: raw real vs matched real vs vacc ─────────────────────
if SAVE_FIGS:
    os.makedirs(os.path.join(os.getcwd(), FIGS_DIR), exist_ok=True)
sensors = range(6) if SENSOR_SUBSET is None else SENSOR_SUBSET
t = np.arange(W)
nfig = 0
for si in sensors:
    for ax, acomp in enumerate("XYZ"):
        fig = plt.figure(figsize=(10, 2.6))
        plt.plot(t, out["imu_acc_raw"][:, si, ax].numpy(), "k", lw=1.2, label="real raw")
        plt.plot(t, out["imu_acc"][:, si, ax].numpy(), "C2", lw=0.9, label="real matched")
        plt.plot(t, out["vacc"][:, si, ax].numpy(), "C1", lw=0.9, label="syn vacc")
        plt.title(f"{SENSOR_NAMES[si]} [{si}] \u2014 acceleration {acomp} (m/s\u00b2)")
        plt.xlabel("frame"); plt.ylabel(f"ACC-{acomp}"); plt.legend(ncol=3, fontsize=8); plt.tight_layout()
        if SAVE_FIGS:
            fig.savefig(os.path.join(os.getcwd(), FIGS_DIR, f"{SENSOR_NAMES[si]}_ACC_{acomp}.png"), dpi=150)
        plt.show(); nfig += 1
print(f"done: {nfig} figures")


In [ ]:
# ── ORI plots: real ori vs FK grot (rotvec, rad) ───────────────────────
for si in sensors:
    for ax, acomp in enumerate("XYZ"):
        fig = plt.figure(figsize=(10, 2.6))
        plt.plot(t, ori_rr[:, si, ax].numpy(), "k", lw=1.2, label="real ori")
        plt.plot(t, ori_syn[:, si, ax].numpy(), "C0", lw=0.9, label="FK grot")
        plt.title(f"{SENSOR_NAMES[si]} [{si}] \u2014 bone orientation rotvec {acomp} (rad)")
        plt.xlabel("frame"); plt.ylabel(f"ORI-{acomp}"); plt.legend(ncol=2, fontsize=8); plt.tight_layout()
        if SAVE_FIGS:
            fig.savefig(os.path.join(os.getcwd(), FIGS_DIR, f"{SENSOR_NAMES[si]}_ORI_{acomp}.png"), dpi=150)
        plt.show(); nfig += 1
print(f"done: {nfig} figures total")


## Reading the comparison

- The **bias-shift table** (cell 3) is upstream's whole gravity/bias story: `vacc-vs-raw` RMSE is dominated by it, `vacc-vs-matched` shows the remaining **dynamics** gap.
- If `vacc-vs-matched` is small but HModel-syn-vs-real was huge, the difference is methodology (bias-match + AMASS shape/frame handling), not motion content.
- The **geodesic table** shows the orientation gap this port does *not* reconcile (`imu_ori` stays real): persistent tens-of-degrees offsets point at mount (`RSB`) / world-frame alignment, matching the earlier cross-subject finding.
- Narrow the window or set `SENSOR_SUBSET` / `SAVE_FIGS=True` to inspect events.